## Evaluation strategy (no ground-truth completions)

The practice set contains **47 completion points without ground-truth** (they are hidden — evaluation happens server-side). Instead we use two **proxy metrics** strongly correlated with ChrF:

1. **Identifier Hit Rate (IHR)** — what fraction of unique identifiers visible in the FIM window appear anywhere in the selected context. Higher IHR → the model has access to more needed tokens → higher expected ChrF.

2. **Import Coverage (IC)** — what fraction of module names from `import`/`from...import` statements in the prefix are represented by at least one file in the context. Measures whether the model sees the dependencies it will need.

3. **Context utilisation** — average number of injected files and average characters used out of the 24,000-character budget.


In [6]:
import sys, os, re, math, zipfile, ast
import jsonlines
from collections import defaultdict
import importlib.util


sys.path.insert(0, os.path.abspath('.'))

print('Working dir:', os.getcwd())

Working dir: c:\Users\Ricardo\Programming\EnsembleAI_Krety_T2\core


In [7]:
DATA_FILE = 'data/python-practice.jsonl'

datapoints = list(jsonlines.open(DATA_FILE))
print(f'Loaded {len(datapoints)} completion points')
print('Keys:', list(datapoints[0].keys()))

Wczytano 47 punktów ukończenia
Klucze: ['id', 'repo', 'revision', 'path', 'modified', 'prefix', 'suffix', 'archive']


In [8]:

def ensure_repo_extracted(archive_name: str) -> str:
    zip_path = os.path.join('data', 'python-practice', archive_name)
    dir_name = archive_name.replace('.zip', '')
    extract_path = os.path.join('data', 'python-practice', dir_name)
    
    if os.path.isdir(extract_path):
        return extract_path
    if os.path.isfile(zip_path):
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        print(f'  Extracted: {archive_name}')
        return extract_path
    return None

repo_dirs = {}
for dp in datapoints:
    archive = dp.get('archive')
    if archive and archive not in repo_dirs:
        path = ensure_repo_extracted(archive)
        repo_dirs[archive] = path

print(f'\nResolved {len(repo_dirs)} unique repositories')


Rozwiązano 47 unikalnych repozytoriów


In [9]:
PYTHON_STOPWORDS = {
    "import", "from", "return", "class", "def", "self", "pass", "True",
    "False", "None", "print", "with", "open", "raise", "assert", "yield",
    "lambda", "global", "nonlocal", "async", "await", "else", "elif",
    "while", "break", "continue", "except", "finally", "try", "for",
    "isinstance", "hasattr", "getattr", "setattr", "super", "type",
    "list", "dict", "tuple", "set", "str", "int", "float", "bool",
    "range", "enumerate", "zip", "map", "filter", "len", "any", "all",
}

def extract_identifiers(text: str) -> set:
    return set(re.findall(r'[a-zA-Z_]\w{2,}', text)) - PYTHON_STOPWORDS

def extract_imports(text: str) -> set:
    mods = set()
    for m in re.finditer(r'^(?:from\s+([\w.]+)\s+import|import\s+([\w., ]+))', text, re.MULTILINE):
        raw = m.group(1) or m.group(2)
        for part in raw.replace(',', ' ').split():
            if part and part != 'as':
                mods.update(part.replace('.', '/').split('/'))
    return {m.lower() for m in mods if len(m) > 1}

def compute_ihr(context_str: str, local_text: str) -> float:
    needed = extract_identifiers(local_text)
    if not needed:
        return 0.0
    return sum(1 for id_ in needed if id_ in context_str) / len(needed)

def compute_ic(context_str: str, prefix: str) -> float:
    mods = extract_imports(prefix)
    if not mods:
        return 1.0
    return sum(1 for m in mods if m in context_str.lower()) / len(mods)

print('Proxy metrics defined.')

Metryki proxy zdefiniowane.


In [10]:

def load_engine(path):
    spec = importlib.util.spec_from_file_location('_engine', path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

eng_base   = load_engine('engine.py')
eng_improv = load_engine('engine_improv.py')

print('Engines loaded.')

Silniki wczytane.


usage: ipykernel_launcher.py [-h] [--stage STAGE] [--lang LANG]
                             [--strategy STRATEGY] [--trim-prefix]
                             [--trim-suffix] [--limit LIMIT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Ricardo\AppData\Roaming\jupyter\runtime\kernel-v34fcadf011cffc01ab766efdd638351eb927cf34d.json
usage: ipykernel_launcher.py [-h] [--stage STAGE] [--lang LANG]
                             [--strategy STRATEGY] [--trim-prefix]
                             [--trim-suffix] [--limit LIMIT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Ricardo\AppData\Roaming\jupyter\runtime\kernel-v34fcadf011cffc01ab766efdd638351eb927cf34d.json


In [11]:

base_caches   = {}
improv_caches = {}

for archive, repo_dir in repo_dirs.items():
    if repo_dir and os.path.isdir(repo_dir):
        print(f'Building cache for {archive}...')
        base_caches[archive]   = eng_base.build_repo_cache(repo_dir)
        improv_caches[archive] = eng_improv.build_repo_cache(repo_dir)

print('\nDone — all repositories cached.')

Buduję cache dla celery__kombu-0d3b1e254f9178828f62b7b84f0307882e28e2a0.zip...
Buduję cache dla celery__kombu-180aabec1ffc71fd802a28231931ba00bf295df5.zip...
Buduję cache dla celery__kombu-31adb300dc794012c404bf73afaa7ecf66d3d07d.zip...
Buduję cache dla celery__kombu-333dfa3cd010a164df89cf0685da1ea5ad657f0c.zip...
Buduję cache dla celery__kombu-75d781ef6164f620f4d87c01ab7b6a79c16b6cc7.zip...
Buduję cache dla celery__kombu-7ccec0b5369f94c51bdf487ac274a68c4b9bdfb9.zip...
Buduję cache dla celery__kombu-7f9674419b585921b1da4ecbd5f3dc203891955e.zip...
Buduję cache dla celery__kombu-844d8d0673b8fa91303ee3b1700f93422afbc6ba.zip...
Buduję cache dla celery__kombu-8ddfb92557b00e4be2ef913ff47ce9ebc0837878.zip...
Buduję cache dla celery__kombu-b304f93ccbea54ea2e9c8c4dccfe3bf8b9256b88.zip...
Buduję cache dla celery__kombu-d570d6a5e213323f940be0e754dfb254a6190c8a.zip...
Buduję cache dla celery__kombu-d57dde5631c5c7dd73300a79613975531112aae6.zip...
Buduję cache dla celery__kombu-da0972ec2003a5c9d59f3

In [12]:
results = []

for dp in datapoints:
    archive = dp.get('archive')
    repo_dir = repo_dirs.get(archive)
    if not repo_dir or not os.path.isdir(repo_dir):
        continue

    prefix   = dp.get('prefix') or ''
    suffix   = dp.get('suffix') or ''

    local_text = '\n'.join(prefix.split('\n')[-30:]) + '\n' + '\n'.join(suffix.split('\n')[:30])

    ctx_base,   n_base   = eng_base.get_context(dp, base_caches[archive])
    ctx_improv, n_improv = eng_improv.get_context(dp, improv_caches[archive], repo_dir)

    results.append({
        'id':          dp.get('id', ''),
        'path':        dp.get('path', ''),
        'archive':     archive,
        'base_ihr':    compute_ihr(ctx_base,   local_text),
        'base_ic':     compute_ic(ctx_base,    prefix),
        'base_len':    len(ctx_base),
        'base_n':      n_base,
        'improv_ihr':  compute_ihr(ctx_improv, local_text),
        'improv_ic':   compute_ic(ctx_improv,  prefix),
        'improv_len':  len(ctx_improv),
        'improv_n':    n_improv,
    })

print(f'Evaluated {len(results)} completion points.')

Oceniono 47 punktów ukończenia.


## Aggregate Results

In [13]:
import statistics

def avg(lst): return sum(lst) / len(lst) if lst else 0

base_ihrs   = [r['base_ihr']   for r in results]
improv_ihrs = [r['improv_ihr'] for r in results]
base_ics    = [r['base_ic']    for r in results]
improv_ics  = [r['improv_ic']  for r in results]

print('=' * 55)
print(f'{"Metric":<35} {"Baseline":>8} {"Improved":>9}')
print('=' * 55)
print(f'{"Identifier Hit Rate (IHR)":<35} {avg(base_ihrs):>8.4f} {avg(improv_ihrs):>9.4f}')
print(f'{"Import Coverage (IC)":<35} {avg(base_ics):>8.4f} {avg(improv_ics):>9.4f}')
print(f'{"Avg context length (chars)":<35} {avg([r["base_len"] for r in results]):>8.0f} {avg([r["improv_len"] for r in results]):>9.0f}')
print(f'{"Avg number of files":<35} {avg([r["base_n"] for r in results]):>8.2f} {avg([r["improv_n"] for r in results]):>9.2f}')
print('=' * 55)

deltas    = [r['improv_ihr'] - r['base_ihr'] for r in results]
improved  = sum(1 for d in deltas if d > 0)
same      = sum(1 for d in deltas if d == 0)
worse     = sum(1 for d in deltas if d < 0)
n         = len(results)

ihr_delta = avg(deltas)
ic_delta  = avg(improv_ics) - avg(base_ics)

print(f'\n  Delta IHR : {ihr_delta:+.4f}   median: {statistics.median(deltas):+.4f}')
print(f'  Delta IC  : {ic_delta:+.4f}')
print(f'  Better    : {improved}/{n} ({100*improved/n:.1f}%)')
print(f'  Same      : {same}/{n}')
print(f'  Worse     : {worse}/{n} ({100*worse/n:.1f}%)')

Metryka                             Baseline Ulepszony
Identifier Hit Rate (IHR)             0.7287    0.7894
Import Coverage (IC)                  0.9048    0.9783
Srednia dlugosc kontekstu (znaki)      21712     23927
Srednia liczba plikow                   6.51     10.66

  Delta IHR : +0.0607   mediana: +0.0233
  Delta IC  : +0.0735
  Lepszy    : 26/47 (55.3%)
  Taki sam  : 8/47
  Gorszy    : 13/47 (27.7%)


## Per-Point Results (sorted by IHR improvement)

In [14]:
sorted_res = sorted(results, key=lambda r: r['improv_ihr'] - r['base_ihr'], reverse=True)

print(f'{"Path":<45} {"IHR base":>9} {"IHR imp":>9} {"Delta":>7}')
print('-' * 73)
for r in sorted_res:
    delta = r['improv_ihr'] - r['base_ihr']
    sign  = '+' if delta >= 0 else ''
    print(f"{r['path'][:44]:<45} {r['base_ihr']:>9.4f} {r['improv_ihr']:>9.4f} {sign}{delta:>6.4f}")

Sciezka                                        IHR base   IHR imp   Delta
-------------------------------------------------------------------------
t/unit/transport/test_filesystem.py              0.3333    0.8431 +0.5098
kombu/tests/test_pidbox.py                       0.5263    1.0000 +0.4737
tests/test_parsing.py                            0.3077    0.7692 +0.4615
kombu/transport/mongodb.py                       0.5125    0.8250 +0.3125
garlicsim/test_garlicsim/test_general_misc/t     0.7105    1.0000 +0.2895
t/integration/test_redis.py                      0.5823    0.8228 +0.2405
werkzeug/contrib/fixers.py                       0.5577    0.7692 +0.2115
tests/test_wsgi.py                               0.6383    0.8298 +0.1915
werkzeug/serving.py                              0.3784    0.5676 +0.1892
werkzeug/debug/repr.py                           0.6923    0.8462 +0.1538
werkzeug/testsuite/contrib/iterio.py             0.8387    0.9677 +0.1290
garlicsim/garlicsim/general_misc/conte

## Comparison Charts

In [15]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not available — skipping charts')

if HAS_MPL:
    n_pts = len(results)
    xs    = list(range(n_pts))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Baseline vs Improved — proxy metrics', fontsize=14, fontweight='bold')

    ax = axes[0]
    ax.bar([x - 0.2 for x in xs], base_ihrs,   width=0.4, label='Baseline',  color='#e07070', alpha=0.85)
    ax.bar([x + 0.2 for x in xs], improv_ihrs,  width=0.4, label='Improved', color='#70a8e0', alpha=0.85)
    ax.axhline(avg(base_ihrs),   color='#c03030', linestyle='--', linewidth=1.2, label=f'Base avg={avg(base_ihrs):.3f}')
    ax.axhline(avg(improv_ihrs), color='#2060c0', linestyle='--', linewidth=1.2, label=f'Impr avg={avg(improv_ihrs):.3f}')
    ax.set_title('Identifier Hit Rate (IHR)')
    ax.set_xlabel('Point index')
    ax.set_ylabel('IHR (higher = better context)')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

    ax2 = axes[1]
    ax2.bar([x - 0.2 for x in xs], base_ics,   width=0.4, label='Baseline',  color='#e07070', alpha=0.85)
    ax2.bar([x + 0.2 for x in xs], improv_ics,  width=0.4, label='Improved', color='#70a8e0', alpha=0.85)
    ax2.axhline(avg(base_ics),   color='#c03030', linestyle='--', linewidth=1.2, label=f'Base avg={avg(base_ics):.3f}')
    ax2.axhline(avg(improv_ics), color='#2060c0', linestyle='--', linewidth=1.2, label=f'Impr avg={avg(improv_ics):.3f}')
    ax2.set_title('Import Coverage (IC)')
    ax2.set_xlabel('Point index')
    ax2.set_ylabel('IC (higher = better import coverage)')
    ax2.set_ylim(0, 1.05)
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('engine_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Chart saved: engine_comparison.png')

Wykres zapisany: engine_comparison.png


C:\Users\Ricardo\AppData\Local\Temp\ipykernel_7024\1857819702.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
if HAS_MPL:
    fig2, ax3 = plt.subplots(figsize=(7, 7))
    ax3.scatter(base_ihrs, improv_ihrs, alpha=0.7, s=50, color='#4070b0')
    lo, hi = 0, 1
    ax3.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='no change')
    ax3.fill_between([lo, hi], [lo, hi], [hi, hi], alpha=0.07, color='green', label='improvement zone')
    ax3.fill_between([lo, hi], [lo, lo], [lo, hi], alpha=0.07, color='red',   label='regression zone')
    ax3.set_xlabel('IHR Baseline')
    ax3.set_ylabel('IHR Improved')
    ax3.set_title('IHR per point: Baseline vs Improved')
    ax3.legend()
    plt.tight_layout()
    plt.savefig('engine_scatter.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Scatter chart saved: engine_scatter.png')

Wykres rozrzutu zapisany: engine_scatter.png


C:\Users\Ricardo\AppData\Local\Temp\ipykernel_7024\644091042.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
improved_n = sum(1 for r in results if r['improv_ihr'] > r['base_ihr'])
same_n     = sum(1 for r in results if r['improv_ihr'] == r['base_ihr'])
worse_n    = sum(1 for r in results if r['improv_ihr'] < r['base_ihr'])
total      = len(results)

print('IHR result per point:')
print(f'  Better  : {improved_n}/{total} ({100*improved_n/total:.1f}%)')
print(f'  Same    : {same_n}/{total} ({100*same_n/total:.1f}%)')
print(f'  Worse   : {worse_n}/{total} ({100*worse_n/total:.1f}%)')
print(f'\n  Delta median  : {statistics.median(deltas):+.4f}')
print(f'  Std dev       : {statistics.stdev(deltas):.4f}')

if HAS_MPL:
    fig3, ax4 = plt.subplots(figsize=(8, 4))
    ax4.hist(deltas, bins=20, color='#5080c0', edgecolor='white')
    ax4.axvline(0, color='red',   linestyle='--', label='no change')
    ax4.axvline(statistics.mean(deltas), color='green', linestyle='-',
                label=f'mean={statistics.mean(deltas):+.3f}')
    ax4.set_xlabel('Delta IHR (Improved - Baseline)')
    ax4.set_ylabel('Number of points')
    ax4.set_title('IHR improvement distribution per point')
    ax4.legend()
    plt.tight_layout()
    plt.savefig('engine_delta_hist.png', dpi=120, bbox_inches='tight')
    plt.show()

C:\Users\Ricardo\AppData\Local\Temp\ipykernel_7024\716998109.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Wynik IHR na punkt:
  Lepszy  : 26/47 (55.3%)
  Taki sam: 8/47 (17.0%)
  Gorszy  : 13/47 (27.7%)

  Mediana delty : +0.0233
  Odch. std     : 0.1637


## Best Case — Qualitative Analysis

In [18]:
best  = max(results, key=lambda r: r['improv_ihr'] - r['base_ihr'])
worst = min(results, key=lambda r: r['improv_ihr'] - r['base_ihr'])

print('Largest IHR gain:')
print(f"  File      : {best['path']}")
print(f"  Baseline  : {best['base_ihr']:.4f}  ({best['base_n']} files)")
print(f"  Improved  : {best['improv_ihr']:.4f}  ({best['improv_n']} files)")
print(f"  Delta     : {best['improv_ihr'] - best['base_ihr']:+.4f}")

print()
print('Largest IHR regression:')
print(f"  File      : {worst['path']}")
print(f"  Baseline  : {worst['base_ihr']:.4f}  ({worst['base_n']} files)")
print(f"  Improved  : {worst['improv_ihr']:.4f}  ({worst['improv_n']} files)")
print(f"  Delta     : {worst['improv_ihr'] - worst['base_ihr']:+.4f}")

Najwiekszy zysk IHR:
  Plik      : t/unit/transport/test_filesystem.py
  Baseline  : 0.3333  (5 plikow)
  Ulepszony : 0.8431  (12 plikow)
  Delta     : +0.5098

Najwiekszy regres IHR:
  Plik      : garlicsim_wx/garlicsim_wx/general_misc/junk/aui.py
  Baseline  : 0.7656  (7 plikow)
  Ulepszony : 0.4375  (18 plikow)
  Delta     : -0.3281


## Results Interpretation

| Improvement | Mechanism | Expected impact on ChrF |
|---|---|---|
| **Level 0: same-dir files** | Files from the same folder get the highest priority slot | **High** — model gets exact local context |
| **AST definition extraction** | More accurate than regex (handles nested defs, decorators) | **Medium** — fewer false positives at level 1 |
| **BM25 ranking (level 3)** | Rare/specific identifiers carry higher weight | **Medium** — better signal-to-noise ratio |
| **Import detection (level 2)** | Library files are retrieved by module names | **Medium** — model sees the dependencies |
| **Project map** | Compact folder tree appended at the end | **Low** — helps the model understand project structure |
| **Wider FIM window (30 vs 20 lines)** | More code around the cursor for vocabulary extraction | **Low-Medium** — captures identifiers further from the cursor |
| **Left-trim defence** | Most important context placed at the bottom (right end) of the string | **High** — survives model context-window truncation |

### Key observation

The first version of the improved engine placed import detection at **level 0** — and IHR DROPPED from 0.73 to 0.52 because library files displaced neighbouring test files. **Locality beats the dependency graph** when selecting context for code completion.
